# Treinamento no Google Colab — CNN (PyTorch) para Imagens Intraorais

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taynaramos/dental-image-classifier/blob/feature/colab-training/experiments/colab_training.ipynb)

Versão do pipeline de treino preparada para rodar no **Google Colab com GPU**, reaproveitando o código de `src/pytorch_classifier` — o comportamento é idêntico ao da CLI (`torch-train`) e ao notebook local (`pytorch_training.ipynb`).

**Antes de executar:**
1. `Ambiente de execução → Alterar tipo de ambiente de execução → GPU` (no Colab Pro, prefira **A100** ou **L4**).
2. Tenha o dataset no seu Google Drive: a pasta com uma sub-pasta por sujeito (ou um `.zip` dela). Ajuste o caminho `DRIVE_DATASET` na célula de dataset.

Todas as dependências do projeto (PyTorch com CUDA, torchvision, scikit-learn, matplotlib) já vêm pré-instaladas no Colab — **não** rode `pip install -r requirements.txt` aqui, pois as versões pinadas substituiriam o PyTorch com suporte a GPU do Colab.

## 1. Clonar o repositório

In [ ]:
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/taynaramos/dental-image-classifier.git"
    BRANCH = "feature/colab-training"  # troque para "main" após o merge
    PROJECT_ROOT = Path("/content/dental-image-classifier")
    if not PROJECT_ROOT.exists():
        !git clone --branch {BRANCH} {REPO_URL} {PROJECT_ROOT}
else:
    # Permite executar este mesmo notebook localmente, a partir de experiments/
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Projeto em: {PROJECT_ROOT}")

## 2. Dataset a partir do Google Drive

O dataset é copiado do Drive para o disco local da VM (`data/dataset`) — ler direto do mount do Drive é muito mais lento durante o treino. Aceita tanto uma **pasta** (uma sub-pasta por sujeito) quanto um **`.zip`** dela.

In [ ]:
import shutil
import zipfile

DATASET_ROOT = PROJECT_ROOT / "data" / "dataset"

if IN_COLAB and not DATASET_ROOT.exists():
    from google.colab import drive
    drive.mount("/content/drive")

    # >>> AJUSTE AQUI: caminho do dataset no seu Drive (pasta ou .zip) <<<
    DRIVE_DATASET = Path("/content/drive/MyDrive/dental-image-classifier/dataset")

    if not DRIVE_DATASET.exists():
        raise FileNotFoundError(
            f"Dataset não encontrado em {DRIVE_DATASET} — ajuste DRIVE_DATASET nesta célula."
        )

    DATASET_ROOT.parent.mkdir(parents=True, exist_ok=True)
    if DRIVE_DATASET.suffix == ".zip":
        with zipfile.ZipFile(DRIVE_DATASET) as zf:
            zf.extractall(DATASET_ROOT)
        # Se o zip tiver uma única pasta raiz, desaninha para DATASET_ROOT
        conteudos = list(DATASET_ROOT.iterdir())
        if len(conteudos) == 1 and conteudos[0].is_dir():
            raiz_unica = conteudos[0]
            for filho in raiz_unica.iterdir():
                shutil.move(str(filho), DATASET_ROOT)
            raiz_unica.rmdir()
    else:
        shutil.copytree(DRIVE_DATASET, DATASET_ROOT)

n_sujeitos = sum(1 for p in DATASET_ROOT.iterdir() if p.is_dir())
print(f"Dataset em : {DATASET_ROOT}")
print(f"Sujeitos   : {n_sujeitos}")

## 3. Importações e verificação da GPU

In [ ]:
import random

import matplotlib.pyplot as plt
import torch
from PIL import Image
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

from src.pytorch_classifier.dataset import build_dataloaders, resolve_imagefolder_root
from src.pytorch_classifier.model import DentalCNN, ModelConfig
from src.pytorch_classifier.predict import predict
from src.pytorch_classifier.trainer import Trainer
from src.pytorch_classifier.utils import get_device, save_checkpoint, set_seed

%matplotlib inline

device = get_device()
print(f"Dispositivo: {device}")
if device.type == "cuda":
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
else:
    print("AVISO: sem GPU — selecione um ambiente de execução com GPU para acelerar o treino.")

## 4. Configuração

Mesmos hiperparâmetros do notebook local; `BATCH_SIZE` e `NUM_WORKERS` maiores para aproveitar a GPU.

In [ ]:
MODEL_OUT = PROJECT_ROOT / "artifacts" / "torch_model.pth"

IMAGE_SIZE    = 128
GRAYSCALE     = True
BATCH_SIZE    = 64
NUM_WORKERS   = 2
EPOCHS        = 20
PATIENCE      = 5      # early stopping: nº de épocas sem melhora na val loss antes de parar
MIN_DELTA     = 0.0    # melhora mínima na val loss para zerar a paciência
LEARNING_RATE = 1e-3
SEED          = 42

set_seed(SEED)
print(f"Modelo (out): {MODEL_OUT}")

## 5. Preparo do dataset e DataLoaders

In [ ]:
# Materializa (uma única vez) o dataset por sujeito em train/val/test por classe.
# A divisão é feita por sujeito, para não vazar dados do mesmo paciente entre conjuntos.
imagefolder_root = resolve_imagefolder_root(DATASET_ROOT, seed=SEED)

train_loader, val_loader, test_loader, classes = build_dataloaders(
    imagefolder_root,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    grayscale=GRAYSCALE,
    num_workers=NUM_WORKERS,
)
NUM_CLASSES = len(classes)

print(f"Classes ({NUM_CLASSES}): {classes}")
print(f"Batches — treino: {len(train_loader)}  val: {len(val_loader)}  teste: {len(test_loader)}")

## 6. Visualização de algumas imagens

In [ ]:
classes_preview = sorted(p.name for p in (imagefolder_root / "train").iterdir() if p.is_dir())

fig, axes = plt.subplots(1, len(classes_preview), figsize=(15, 3))
for ax, classe in zip(axes, classes_preview):
    exemplo = random.choice(list((imagefolder_root / "train" / classe).glob("*")))
    ax.imshow(Image.open(exemplo))
    ax.set_title(classe, fontsize=9)
    ax.axis('off')
fig.suptitle("Um exemplo aleatório por classe (conjunto de treino)", y=1.02)
plt.tight_layout()
plt.show()

## 7. Modelo e treinamento

In [ ]:
config = ModelConfig(image_size=IMAGE_SIZE, grayscale=GRAYSCALE)
model = DentalCNN(num_classes=NUM_CLASSES, in_channels=config.in_channels)

total_params = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis: {total_params:,}")

trainer = Trainer(model, device, learning_rate=LEARNING_RATE)
history = trainer.fit(train_loader, val_loader, epochs=EPOCHS, patience=PATIENCE, min_delta=MIN_DELTA)

## 8. Curvas de Loss e Acurácia

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.train_loss, label="Treino")
ax1.plot(history.val_loss, label="Validação")
ax1.set_xlabel("Época")
ax1.set_ylabel("Loss")
ax1.set_title("Curva de Loss")
ax1.legend()

ax2.plot(history.train_accuracy, label="Treino")
ax2.plot(history.val_accuracy, label="Validação")
ax2.set_xlabel("Época")
ax2.set_ylabel("Acurácia")
ax2.set_title("Curva de Acurácia")
ax2.legend()

plt.tight_layout()
plt.show()

## 9. Avaliação final e matriz de confusão

In [ ]:
test_loss, test_acc = trainer.evaluate(test_loader)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc:.4f}\n")

model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        y_pred.extend(outputs.argmax(dim=1).cpu().tolist())
        y_true.extend(labels.tolist())

print(classification_report(y_true, y_pred, target_names=classes))

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    y_true, y_pred, display_labels=classes, cmap="Blues", xticks_rotation=45, ax=ax,
)
ax.set_title(f"Matriz de Confusão — Teste (acurácia = {test_acc:.4f})")
plt.tight_layout()
plt.show()

## 10. Predição em uma imagem de exemplo

In [ ]:
classe_exemplo = random.choice(classes)
exemplo_path = random.choice(list((imagefolder_root / "test" / classe_exemplo).glob("*")))
pred = predict(exemplo_path, model, classes, config, device)

plt.figure(figsize=(3, 3))
plt.imshow(Image.open(exemplo_path))
plt.title(f"Previsto: {pred.label}")
plt.axis('off')
plt.show()

print(f"Imagem          : {exemplo_path.name}")
print(f"Classe real     : {classe_exemplo}")
print(f"Classe prevista : {pred.label}")
print("Probabilidades  :")
for cls, prob in sorted(pred.probabilities.items(), key=lambda kv: -kv[1]):
    print(f"  {cls:<20} {prob:.1%}")

## 11. Salvar o modelo (e copiar para o Drive)

A VM do Colab é descartada ao fim da sessão — o checkpoint é copiado para o Drive para não ser perdido.

In [ ]:
save_checkpoint(MODEL_OUT, model, classes, config)
print(f"Modelo salvo em: {MODEL_OUT}")

if IN_COLAB:
    DRIVE_OUT = Path("/content/drive/MyDrive/dental-image-classifier/artifacts")
    DRIVE_OUT.mkdir(parents=True, exist_ok=True)
    shutil.copy2(MODEL_OUT, DRIVE_OUT / MODEL_OUT.name)
    print(f"Cópia no Drive : {DRIVE_OUT / MODEL_OUT.name}")